# Aula 04: Fontes de Dados, Qualidade e Seleção com `loc`/`iloc`

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

1. **Identificar** as principais fontes e formatos de dados usados em Ciência de Dados.
2. **Carregar** dados de diferentes origens com o `pandas` (CSV, Excel, JSON, SQL, Parquet).
3. **Avaliar** a qualidade de um conjunto de dados usando dimensões objetivas (completude, unicidade, validade, consistência, acurácia e atualidade).
4. **Selecionar** subconjuntos de dados com precisão usando `loc`, `iloc`, indexação booleana e `query`.

> **Por que isso é importante?** Estudos da área estimam que *cientistas de dados gastam de 60% a 80% do tempo* coletando, entendendo e limpando dados. A frase *"garbage in, garbage out"* resume a ideia: nenhum modelo, por mais sofisticado, corrige dados ruins na entrada. Dominar a **aquisição** e a **avaliação de qualidade** é, portanto, a base de todo o restante do curso.

---

## 1. Fontes de dados: de onde vêm os dados?

Na prática, os dados chegam de origens muito diferentes. Vale conhecer as principais categorias:

| Categoria | Exemplos | Como acessamos |
|---|---|---|
| **Arquivos locais** | CSV, TSV, Excel, JSON, Parquet | `pd.read_csv`, `pd.read_excel`, `pd.read_json`, `pd.read_parquet` |
| **Bancos de dados** | PostgreSQL, MySQL, SQLite, SQL Server | `pd.read_sql` + conexão (SQLAlchemy) |
| **APIs / Web** | REST APIs, JSON de serviços web | `requests` + `pd.json_normalize` |
| **Web scraping** | Tabelas em páginas HTML | `pd.read_html`, `BeautifulSoup` |
| **Datasets prontos** | `seaborn`, `sklearn`, OpenML, Kaggle | `sns.load_dataset`, `fetch_openml` |
| **Streaming / Big Data** | Kafka, Spark, data lakes | conectores específicos |

Também classificamos os dados quanto à **estrutura**:

- **Estruturados**: tabelas com linhas e colunas bem definidas (o foco do `pandas`).
- **Semiestruturados**: JSON, XML, logs (têm estrutura, mas flexível).
- **Não estruturados**: texto livre, imagens, áudio, vídeo.


### Formatos de arquivo mais comuns

| Formato | Vantagens | Cuidados | Leitura no pandas |
|---|---|---|---|
| **CSV** | universal, legível, simples | sem tipos, sem compressão, separador/encoding variam | `pd.read_csv` |
| **Excel (.xlsx)** | popular em negócios, múltiplas abas | lento, formatação atrapalha | `pd.read_excel` |
| **JSON** | flexível, ideal para APIs | aninhado exige normalização | `pd.read_json` / `pd.json_normalize` |
| **Parquet** | colunar, comprimido, preserva tipos | binário (não legível a olho nu) | `pd.read_parquet` |
| **SQL** | consultas poderosas, dados sempre atuais | exige conexão e credenciais | `pd.read_sql` |

Os arquivos desta aula já estão na pasta `dados/` do projeto.


In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

In [ ]:
DADOS = "dados/"

In [ ]:
# 1) CSV no padrão brasileiro: ; com separador, vírgula decimal, encoding latin-1
df_csv = pd.read_csv(
    DADOS + "vendas.csv",
    sep=";",
    decimal=",",
    encoding="latin-1",
    na_values=["NA", "-", "?", ""],
    parse_dates=["data"],
)
print("CSV:", df_csv.shape, "| missing em cidade:", df_csv["cidade"].isna().sum())
df_csv.head()

In [ ]:
# 2) Excel — aba específica
df_xlsx = pd.read_excel(DADOS + "planilha.xlsx", sheet_name="Vendas")
print("Excel (aba Vendas):", df_xlsx.shape)
df_xlsx.head()

In [ ]:
# 3) JSON simples
df_json = pd.read_json(DADOS + "clientes.json")
print("JSON simples:", df_json.shape)
df_json.head()

In [ ]:
pedidos = json.loads(Path(DADOS + "pedidos.json").read_text(encoding="utf-8"))
df_itens = pd.json_normalize(pedidos, record_path="itens", meta=["loja", "cidade"])
print("JSON aninhado (itens):")
print(df_itens)

In [ ]:
# 5) SQLite (arquivo local, sem servidor)
conn = sqlite3.connect(DADOS + "clientes.db")
df_sql = pd.read_sql("SELECT * FROM clientes WHERE ativo = 1", conn)
conn.close()
print("SQL (clientes ativos):", df_sql.shape)
df_sql.head()

In [ ]:
# 6) Parquet — preserva tipos (data já vem como datetime)
df_parquet = pd.read_parquet(DADOS + "vendas.parquet")
print("Parquet dtypes:", dict(df_parquet.dtypes))
df_parquet.head()

In [ ]:
# 7) Tabela HTML local (equivalente a pd.read_html de uma página)
df_html = pd.read_html(DADOS + "tabela.html")[0]
print("HTML:")
df_html.head()

In [ ]:
import pandas as pd
import seaborn as sns

# Dataset didático "Titanic": cada linha é um passageiro; queremos entender
# fatores associados à sobrevivência. É um bom caso porque TEM problemas de
# qualidade reais (valores ausentes, colunas redundantes, inconsistências).
df = sns.load_dataset('titanic')

print('Dimensões (linhas, colunas):', df.shape)
print('Colunas:', list(df.columns))
df.head()

## 2. Qualidade dos dados

Dados de baixa qualidade levam a análises enviesadas e modelos que falham em produção. Avaliamos a qualidade por **dimensões**: cada uma responde a uma pergunta objetiva:

| Dimensão | Pergunta que responde | Como medir no pandas |
|---|---|---|
| **Completude** | Faltam valores? | `df.isnull().mean()` (proporção de missing) |
| **Unicidade** | Há registros duplicados? | `df.duplicated().sum()` |
| **Validade** | Os valores respeitam o formato/domínio esperado? | comparar com regras (ex.: idade ≥ 0) |
| **Consistência** | Os dados se contradizem entre colunas/tabelas? | checar regras cruzadas (ex.: `fare == 0` em 1ª classe) |
| **Acurácia** | Os valores refletem a realidade? | comparar com fonte de referência |
| **Atualidade** | Os dados estão atualizados? | comparar datas com o momento de uso |

Nesta aula focamos nas dimensões que conseguimos medir diretamente da tabela: **completude, unicidade, validade e consistência**.

O primeiro passo prático é sempre um **perfilamento inicial** (*data profiling*): olhar formato, tipos, estatísticas e valores ausentes antes de qualquer análise.


### Checklist rápido de qualidade

Um primeiro diagnóstico responde: *Quantas linhas? Quantos valores faltam? Há duplicatas? Como se distribuem as variáveis?* O `describe(include='all')` resume numéricas **e** categóricas em uma única tabela.

In [ ]:
# Completude: quantos valores faltam por coluna?
print('Linhas:', len(df))
print('\nMissing por coluna:\n', df.isnull().sum())

# Unicidade: existem linhas totalmente repetidas?
print('\nDuplicatas:', df.duplicated().sum())

In [ ]:
# Perfil estatístico de TODAS as colunas (numéricas + categóricas)
df.describe(include='all')

In [ ]:
# Perfilamento aprofundado: tipos, cardinalidade, % de missing e checagens de validade

# a) Tipos de dados e uso de memória
print('--- Tipos de dados ---')
print(df.dtypes)

# b) Proporção de valores ausentes por coluna (ordenada)
print('\n--- % de valores ausentes ---')
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

# c) Cardinalidade das colunas categóricas (nº de valores distintos)
print('\n--- Valores distintos (categóricas) ---')
print(df.select_dtypes(include=['object', 'category']).nunique())

# d) Distribuição de uma categórica
print('\n--- Distribuição de "class" ---')
print(df['class'].value_counts(dropna=False))

# e) VALIDADE/CONSISTÊNCIA: regras de negócio (devem retornar 0)
print('\n--- Checagens de validade ---')
print('Idades negativas:', (df['age'] < 0).sum())
print('Tarifas (fare) negativas:', (df['fare'] < 0).sum())
print('Tarifa = 0 (possível inconsistência):', (df['fare'] == 0).sum())

## 3. Seleção de dados: `loc`, `iloc`, indexação booleana e `query`

Selecionar exatamente as linhas e colunas certas é uma habilidade diária. As quatro formas principais:

| Ferramenta | Seleciona por | Exemplo |
|---|---|---|
| **`.loc[]`** | **rótulos** (nomes de índice/colunas) | `df.loc[df['age'] > 30, ['age', 'fare']]` |
| **`.iloc[]`** | **posições inteiras** (0, 1, 2, ...) | `df.iloc[0:5, 0:3]` |
| **Indexação booleana** | condição lógica (`True`/`False`) | `df[df['survived'] == 1]` |
| **`.query()`** | expressão em string (legível) | `df.query('age > 30 and survived == 1')` |

**Pontos de atenção:**

- Em `.loc`, o intervalo é **inclusivo** nas duas pontas (`df.loc[0:5]` traz 6 linhas); em `.iloc`, é **exclusivo** no fim (`df.iloc[0:5]` traz 5 linhas), como o *slicing* padrão do Python.
- Combine condições com `&` (e), `|` (ou), `~` (não) **sempre entre parênteses**: `(cond1) & (cond2)`.
- Para **atribuir** valores a um subconjunto, use sempre `.loc` para evitar o aviso `SettingWithCopyWarning`: `df.loc[df['age'] < 1, 'age'] = None`.


### `loc` vs. `iloc` na prática

A seguir, quatro formas de selecionar os mesmos dados. Observe a diferença entre selecionar por **rótulo** (`loc`) e por **posição** (`iloc`), e como `query` deixa o filtro mais legível.

In [ ]:
# (a) loc: seleção por RÓTULO + condição booleana (parênteses em cada condição!)
sobreviventes_1classe = df.loc[(df['class'] == 'First') & (df['survived'] == 1)]
print('Sobreviventes da 1a classe:', len(sobreviventes_1classe))

# (b) iloc: seleção por POSIÇÃO (10 primeiras linhas, colunas de índice 1 a 4)
print('\nRecorte por posição (iloc):')
print(df.iloc[:10, 1:5])

In [ ]:
# (c) query: mesmo filtro de (a), porém mais legível
print('\nMesmo filtro com query:', len(df.query("`class` == 'First' and survived == 1")))

In [ ]:
# (d) groupby: mini-relatório: tarifa média/mediana e taxa de sobrevivência por classe
resumo = df.groupby('class', observed=True).agg(
    fare_media=('fare', 'mean'),
    fare_mediana=('fare', 'median'),
    taxa_sobrevivencia=('survived', 'mean'),
    n=('survived', 'count'),
)
resumo

### Boas práticas de seleção e agregação

- Prefira `.query()` para filtros longos e legíveis; use indexação booleana para filtros curtos.
- Use `groupby` + agregações para resumir por categoria: `df.groupby('class')['fare'].agg(['mean', 'median', 'count'])`.
- Encadeie operações com `.pipe()` ou quebre em passos nomeados, código legível vale mais que código curto.
- Documente cada filtro: *por que* esse recorte de dados foi escolhido?

Antes do exercício, veja um exemplo de agregação (compare a tarifa média por classe): isso já é um mini-relatório de qualidade e perfil dos dados.


## Exercício: Relatório de qualidade

Produza um pequeno **relatório de qualidade e perfil** do dataset Titanic. Para cada item, escreva o código e um comentário curto interpretando o resultado. Ao final, responda: *o dataset está pronto para análise? O que precisaria ser limpo primeiro?*

In [ ]:
# Seu código a partir daqui:
# ------------------------------------------------------------

# Exemplo do item 1 para começar:
# maiores_60 = df.query('age > 60')
# print('Passageiros com mais de 60 anos:', len(maiores_60))